In [12]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import polars as pl
import plotly.graph_objects as go

In [2]:
def generate_tree(dir_path: Path, prefix: str = ""):
    """
    A recursive generator that yields a visual tree structure of a directory.
    """
    # Visual characters for the tree structure
    space = "    "
    branch = "│   "
    tee = "├── "
    last = "└── "

    # Get directory contents, ignoring hidden files (optional, but standard)
    contents = [path for path in dir_path.iterdir() if not path.name.startswith(".")]

    # Optional: Sort so directories appear first, then files alphabetically
    contents.sort(key=lambda x: (x.is_file(), x.name))

    # Determine the prefix for each item
    pointers = [tee] * (len(contents) - 1) + [last] if contents else []

    for pointer, path in zip(pointers, contents):
        yield prefix + pointer + path.name

        if path.is_dir():
            # If it's a directory, extend the prefix and recurse
            extension = branch if pointer == tee else space
            yield from generate_tree(path, prefix=prefix + extension)
            
def find_project_root(current_path: Path, marker: str = ".git") -> Path:
    """
    Traverse upward through the directory tree until a directory
    containing the specified marker file/folder is found.
    """
    # .parents generates an iterator over all parent directories
    for parent_dir in [current_path] + list(current_path.parents):
        if (parent_dir / marker).exists():
            return parent_dir

    # If the system root is reached without finding it,
    # you can raise an exception or return a default value
    raise FileNotFoundError(f"Could not find the project root containing {marker}!")


def print_list_aligned(
    items: list,
    *,
    vline=True,
    hline=True,
    box=True,
    fixed_width=None,
    min_width=None,
):
    """
    Print a list of items in a vertically aligned format.
    Each item is printed on a new line, and the width of the printed items
    can be adjusted to be fixed or based on the longest item.

    vline: if True, draw vertical bars at left/right of each row.
    hline: if True, draw a horizontal line (top and bottom) around the block.
    """
    # Expected a list of lists, where each inner list represents a row of items to be printed
    assert all(isinstance(item, list) for item in items), "Expected a list of lists!"
    # Determine the maximum width of items in each column
    num_columns = max(len(row) for row in items)
    column_widths = [0] * num_columns
    for row in items:
        for i, item in enumerate(row):
            column_widths[i] = max(column_widths[i], len(str(item)))
    # If fixed_width is provided, override the calculated column widths
    if fixed_width is not None:
        column_widths = [fixed_width] * num_columns
    # If min_width is provided, ensure each column width is at least min_width
    if min_width is not None:
        column_widths = [max(width, min_width) for width in column_widths]

    # Precompute lengths used for horizontal lines
    # Each column prints as: <content ljust(width)> + "  "  (two trailing spaces)
    len_line = sum((w + 2) for w in column_widths)
    # When vline is True, printed row is "│ {line}│" so there's one extra leading space inside the bars
    inner_width = len_line + (1 if vline else 0)

    def _print_top():
        if vline:
            print("┌" + "─" * inner_width + "┐")
        else:
            print("─" * inner_width)

    def _print_bottom():
        if vline:
            print("└" + "─" * inner_width + "┘")
        else:
            print("─" * inner_width)

    # Print top horizontal line if requested
    if box:
        _print_top()

    # Print each row with aligned columns
    for row in items:
        line = ""
        if not vline:
            for i, item in enumerate(row):
                line += (
                    str(item).ljust(column_widths[i]) + "  "
                )  # Add spacing between columns
        else:
            for i, item in enumerate(row[:-1]):
                line += (
                    str(item).ljust(column_widths[i]) + " │ "
                )  # Add vertical bar after each column (except the last one)
            # Add the last column without a trailing vertical bar
            if row:
                line += str(row[-1]).ljust(column_widths[len(row) - 1]) + " "
        if box:
            line = "│ " + line + "│"  # Add vertical bars and trim trailing spaces
        print(line)

    # Print bottom horizontal line if requested
    if hline:
        _print_bottom()

In [3]:
# List root files in the output directory 2 levels up
output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../output"))
print(Path(output_dir))
for line in generate_tree(Path(output_dir)):
    print(line)
files = [f for f in Path(output_dir).rglob("*.root")]

# print(files)

/home/fanghan/Work/RPIL/QMIRT/gate10mc/dev/python/optical_photon_sim/output
├── phase_output.root
└── phsa_output.root


In [5]:
with uproot.open(Path(output_dir) / "phsa_output.root") as file:
    tree = file["crystal_phsa"]
    crystal_phsa_df =  pl.DataFrame(tree.arrays(tree.keys(), library="np"))
    tree = file["world_phsa"]
    world_phsa_df = pl.DataFrame(tree.arrays(tree.keys(), library="np"))
print(crystal_phsa_df)
print(world_phsa_df)

shape: (160, 14)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬─────────┐
│ Position_X ┆ Position_ ┆ Position_ ┆ PostPosit ┆ … ┆ TrackCrea ┆ EventKine ┆ KineticEn ┆ PDGCode │
│ ---        ┆ Y         ┆ Z         ┆ ion_X     ┆   ┆ torProces ┆ ticEnergy ┆ ergy      ┆ ---     │
│ f64        ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s         ┆ ---       ┆ ---       ┆ i32     │
│            ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ f64       ┆ f64       ┆         │
│            ┆           ┆           ┆           ┆   ┆ str       ┆           ┆           ┆         │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═════════╡
│ 0.0        ┆ 0.0       ┆ 9.940267  ┆ 0.0       ┆ … ┆ none      ┆ 0.14      ┆ 0.14      ┆ 22      │
│ 0.0        ┆ 0.0       ┆ 9.619695  ┆ 0.0       ┆ … ┆ none      ┆ 0.14      ┆ 0.14      ┆ 22      │
│ 0.0        ┆ 0.0       ┆ 8.756714  ┆ 0.0       ┆ … ┆ none      ┆ 0.14   

In [7]:
world_PDFCode = world_phsa_df["PDGCode"].to_numpy()
crystal_PDFCode = crystal_phsa_df["PDGCode"].to_numpy()
unique, counts = np.unique(world_PDFCode, return_counts=True)
print("World Unique PDG Codes:", unique)
print("Counts:", counts)
unique, counts = np.unique(crystal_PDFCode, return_counts=True)
print("Crystal Unique PDG Codes:", unique)
print("Counts:", counts)


World Unique PDG Codes: [22]
Counts: [167]
Crystal Unique PDG Codes: [22]
Counts: [160]


In [13]:
# Plot the position distribution of photons in the crystal and in the world
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=world_phsa_df["Position_X"],
        y=world_phsa_df["Position_Y"],
        z=world_phsa_df["Position_Z"],
        mode="markers",
        name="World",
        marker=dict(size=3, color="blue", opacity=0.5),
    )
)
fig.add_trace(
    go.Scatter3d(
        x=crystal_phsa_df["Position_X"],
        y=crystal_phsa_df["Position_Y"],
        z=crystal_phsa_df["Position_Z"],
        mode="markers",
        name="Crystal",
        marker=dict(size=3, color="red", opacity=0.5),
    )
)
fig.update_layout(
    title="Photon Position Distribution",
    scene=dict(
        xaxis_title="Position X (mm)",
        yaxis_title="Position Y (mm)",
        zaxis_title="Position Z (mm)",
    ),
    legend=dict(itemsizing="constant"),
    width=1000,
    height=650,
)

# Add a 6 cm by 6 cm by 6 cm box representing the world volume

fig.show()
